# 情感分析可视化 - 中文版本 (Plotly)

使用问文（Qwen）进行情感分析，用Plotly展示4种主要的可视化结果。

In [7]:
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import numpy as np
import os

# Plotly自动支持中文，无需额外配置

In [2]:
# 读取数据
file_path = 'results/qwen_analysis_results.csv'
df = pd.read_csv(file_path)

# 显示数据框信息
print(f'数据加载成功！')
print(f'总行数: {len(df)}')
print(f'列名: {list(df.columns)}')
print(f'\n前5行数据:')
print(df.head())

数据加载成功！
总行数: 31857
列名: ['post_id', 'time', 'year', 'month', 'text', 'text_length', 'qwen_sentiment', 'qwen_bucket', 'qwen_confidence', 'qwen_reasoning', 'qwen_error', 'qwen_processed_at']

前5行数据:
            post_id                 time  year  month  \
0  3926263409944640  2016-01-01 00:00:00  2016      1   
1  3926263595033328  2016-01-01 00:00:00  2016      1   
2  3926263775522578  2016-01-01 00:01:00  2016      1   
3  3926263683303411  2016-01-01 00:01:00  2016      1   
4  3926266073450184  2016-01-01 00:10:00  2016      1   

                                                text  text_length  \
0  旧年乘霾己离去，钟声敲响新年来。祝福声声随心愿，好人好梦定成真。祝新年，空气新，雾霾少。挣钱...          140   
1  峰峰2016年了看见你在跨年上唱请跟我联系和想你我觉得喜欢上你是一件幸运的事你说妈妈生日快乐...          144   
2  天然第一抗生素——鱼腥草 “鱼腥草”的传说故事 相传很久以前，在一个贫困的村子里，有对不孝夫...          101   
3  2016年了，时间过得好快，长话短说， 16年，我要-瘦身，马甲线 -读很多的书 -认真学习...           89   
4  不知从什么时候开始，觉得每天过得很快，一年也是眨眼的功夫就过完了。2015让我更加感知自我。...          131   

   qwen_sentiment qwen_bucket  qwen_confidence

## 1. 情感分布

显示所有文章按情感类别的分布情况。

In [3]:
if df is not None and 'qwen_sentiment' in df.columns:
    sentiment_counts = df['qwen_sentiment'].value_counts().sort_index()
    sentiment_labels = {-2: '强烈负面', -1: '略微负面', 0: '中立/无关', 1: '略微正面', 2: '强烈正面'}
    sentiment_colors = ['#d62728', '#ff7f0e', '#2ca02c', '#1f77b4', '#9467bd']
    
    # 准备数据
    plot_data = {
        '情感类别': [sentiment_labels[i] for i in sentiment_counts.index],
        '文章数': sentiment_counts.values,
        '百分比': (sentiment_counts.values / sentiment_counts.sum() * 100).round(1)
    }
    
    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=plot_data['情感类别'],
        y=plot_data['文章数'],
        marker=dict(color=sentiment_colors),
        text=[f"{count}<br>{pct:.1f}%" for count, pct in zip(plot_data['文章数'], plot_data['百分比'])],
        textposition='outside',
        hovertemplate='<b>%{x}</b><br>数量: %{y}<extra></extra>'
    ))
    
    fig.update_layout(
        title=dict(text=f'情感分布 (n={len(df)} 篇文章)', font=dict(size=18)),
        xaxis_title='情感类别',
        yaxis_title='文章数量',
        height=600,
        showlegend=False,
        hovermode='x unified',
        font=dict(size=12)
    )
    
    fig.write_image('results/sentiment_distribution_chinese.png', width=1200, height=600)
    print('✓ 图表已保存到: results/sentiment_distribution_chinese.png')

✓ 图表已保存到: results/sentiment_distribution_chinese.png


## 2. 按年份的时间情感分布

显示每年各情感类别的百分比分布。

In [4]:
if df is not None:
    # 检测日期列
    date_col = None
    for col in ['time', 'created_at', 'date', 'timestamp']:
        if col in df.columns:
            date_col = col
            break
    
    if date_col:
        # 转换为日期时间格式
        df_copy = df.copy()
        df_copy[date_col] = pd.to_datetime(df_copy[date_col], errors='coerce')
        df_valid = df_copy[df_copy[date_col].notna()].copy()
        df_valid['year'] = df_valid[date_col].dt.year
        
        # 按年份和情感聚合
        yearly_counts = df_valid.groupby(['year', 'qwen_sentiment']).size().unstack(fill_value=0)
        yearly_pct = yearly_counts.div(yearly_counts.sum(axis=1), axis=0) * 100
        
        sentiment_labels_map = {-2: '强烈负面', -1: '略微负面', 0: '中立/无关', 1: '略微正面', 2: '强烈正面'}
        sentiment_colors_map = {-2: '#d62728', -1: '#ff7f0e', 0: '#2ca02c', 1: '#1f77b4', 2: '#9467bd'}
        
        fig = go.Figure()
        
        for sentiment_val in sorted(yearly_pct.columns):
            fig.add_trace(go.Bar(
                x=yearly_pct.index,
                y=yearly_pct[sentiment_val],
                name=sentiment_labels_map[sentiment_val],
                marker=dict(color=sentiment_colors_map[sentiment_val]),
                hovertemplate='<b>%{x}</b><br>' + sentiment_labels_map[sentiment_val] + ': %{y:.1f}%<extra></extra>'
            ))
        
        fig.update_layout(
            title='按年份的时间情感分布',
            barmode='stack',
            xaxis_title='年份',
            yaxis_title='百分比 (%)',
            height=600,
            hovermode='x unified',
            font=dict(size=12),
            legend=dict(title='情感类别')
        )
        
        fig.write_image('results/temporal_sentiment_distribution_chinese.png', width=1400, height=600)
        print('✓ 图表已保存到: results/temporal_sentiment_distribution_chinese.png')

✓ 图表已保存到: results/temporal_sentiment_distribution_chinese.png


## 3. 内容桶分布

显示不同内容类别的分布。

In [5]:
if df is not None and 'qwen_bucket' in df.columns:
    bucket_counts = df['qwen_bucket'].value_counts()
    
    # 创建子图
    from plotly.subplots import make_subplots
    
    fig = make_subplots(
        rows=1, cols=2,
        specs=[[{'type': 'bar'}, {'type': 'pie'}]],
        subplot_titles=('内容桶分布（柱状图）', '内容桶分布（饼图）')
    )
    
    # 柱状图
    fig.add_trace(
        go.Bar(
            x=bucket_counts.index,
            y=bucket_counts.values,
            marker=dict(color='steelblue'),
            hovertemplate='<b>%{x}</b><br>数量: %{y}<extra></extra>',
            showlegend=False
        ),
        row=1, col=1
    )
    
    # 饼图
    fig.add_trace(
        go.Pie(
            labels=bucket_counts.index,
            values=bucket_counts.values,
            hovertemplate='<b>%{label}</b><br>百分比: %{percent}<extra></extra>',
            textposition='inside',
            textinfo='percent+label'
        ),
        row=1, col=2
    )
    
    fig.update_xaxes(title_text='内容类别', row=1, col=1)
    fig.update_yaxes(title_text='文章数', row=1, col=1)
    
    fig.update_layout(
        title='内容桶分布',
        height=500,
        font=dict(size=11)
    )
    
    fig.write_image('results/bucket_distribution_chinese.png', width=1200, height=500)
    print('✓ 图表已保存到: results/bucket_distribution_chinese.png')

✓ 图表已保存到: results/bucket_distribution_chinese.png


## 4. 情感-内容桶交叉分析热力图

展示情感分布如何在不同内容类别中分布。

In [8]:
if df is not None and 'qwen_bucket' in df.columns:
    relevant_df = df[(df['qwen_sentiment'] != 0) & (df['qwen_bucket'].notna()) & (df['qwen_bucket'] != '')]
    
    if len(relevant_df) > 0:
        # Create crosstab
        sentiment_bucket_crosstab = pd.crosstab(relevant_df['qwen_bucket'], relevant_df['qwen_sentiment'])
        
        # Define sentiment labels
        sentiment_labels_map = {-2: '强烈负面', -1: '略微负面', 0: '中立/无关', 1: '略微正面', 2: '强烈正面'}
        
        # Create heatmap with Plotly
        fig = go.Figure(data=go.Heatmap(
            z=sentiment_bucket_crosstab.values,
            x=[sentiment_labels_map.get(int(col), str(col)) for col in sentiment_bucket_crosstab.columns],
            y=sentiment_bucket_crosstab.index,
            colorscale='Blues',
            text=sentiment_bucket_crosstab.values,
            texttemplate='%{text}',
            textfont={"size": 12},
            hovertemplate='内容桶: %{y}<br>情感: %{x}<br>文章数: %{z}<extra></extra>',
            colorbar=dict(title='文章数')
        ))
        
        fig.update_layout(
            title='情感分布与内容桶交叉分析',
            xaxis_title='情感类别',
            yaxis_title='内容桶',
            height=600,
            width=1000,
            font=dict(size=12)
        )
        
        fig.write_image('results/sentiment_bucket_heatmap_chinese.png', width=1200, height=700)
        print('✓ 图表已保存到: results/sentiment_bucket_heatmap_chinese.png')
    else:
        print('没有找到具有内容桶分类的文章用于热力图')

✓ 图表已保存到: results/sentiment_bucket_heatmap_chinese.png
